# Lab 5: FIR Filter -- MMIO vs DMA Streaming

This notebook compares two approaches for running an FIR filter on the Pynq FPGA:
1. **MMIO**: CPU sends one sample at a time via AXI-Lite (slow)
2. **DMA Streaming**: CPU sets up a bulk transfer, HW processes autonomously (fast)

Both use the same 21-tap low-pass filter.

## 1. Load Overlay

In [36]:
import time
import struct
import numpy as np
from pynq import Overlay, allocate

ol = Overlay("matmulcheck.bit")
print("Overlay loaded")
print("IP blocks:", list(ol.ip_dict.keys()))

dma         = ol.axi_dma_0
matmul_stream  = ol.matmul_stream_2_0
fir_mmio_ip = ol.fir_mmio_0
hw_timer    = ol.axi_timer_0

Overlay loaded
IP blocks: ['matmul_stream_2_0', 'fir_mmio_0', 'axi_timer_0', 'axi_dma_0', 'ps7']


## 2. Prepare Test Signal

50 Hz (passband) + 300 Hz (stopband), sampled at 1 kHz.

In [38]:
import sys
import os
import numpy as np

def normalize_vector(x):
    norm = np.linalg.norm(x)
    return x if norm == 0 else x / norm

def normalize_matrix(A):
    norm = np.linalg.norm(A)
    return A if norm == 0 else A / norm

def generate_vector(n, save=True):
    os.makedirs("test/input", exist_ok=True)
    X = np.random.randn(n, 1)
    X = normalize_vector(X)
    if save:
        path = f"test/input/{n}.txt"
        np.savetxt(path, X, fmt="%.6f")
        print(f"Vector saved to {path}")
    return X

def generate_matrix(n, save=True):
    os.makedirs("test/matrices", exist_ok=True)
    A = np.random.randn(n, n)
    A = normalize_matrix(A)
    if save:
        path = f"test/matrices/{n}.txt"
        np.savetxt(path, A, fmt="%.6f")
        print(f"Matrix saved to {path}")
    return A

def generate_all(n, save=True):
    A = generate_matrix(n, save)
    X = generate_vector(n, save)
    Y = np.matmul(A, X)
    if save:
        os.makedirs("test/output", exist_ok=True)
        out_path = f"test/output/{n}.txt"
        np.savetxt(out_path, Y, fmt="%.6f")
        print(f"Output (Y = AX) saved to {out_path}")
    return A, X, Y

def to_fixed(arr):
    return np.rint(arr.astype(np.float32) * SCALE).astype(np.int32)

def from_fixed(arr):
    return arr.astype(np.int32).astype(np.float32) / SCALE



In [39]:
N               = 10
FRACTIONAL_BITS = 20  # must match config.h
SCALE           = 1 << FRACTIONAL_BITS

matrix_data, input_vector, output_vector = generate_all(N, False)

matrix_data  = np.array(matrix_data,  dtype=np.float32)
input_vector = np.array(input_vector, dtype=np.float32)

print(matrix_data.shape)
print(input_vector.shape)

N_SAMPLES = N*N + N   # matrix + vector

matrix      = matrix_data.reshape(N, N)
x           = input_vector.reshape(N,)
matrix_flat = matrix.flatten()
full_input  = np.concatenate([matrix_flat, x]).astype(np.float32)
full_input_fixed = to_fixed(full_input)

(10, 10)
(10, 1)


## 3. MMIO FIR (one sample per round-trip)

Each sample requires: write input $\rightarrow$ start IP $\rightarrow$ poll done $\rightarrow$ read output.

**Note:** Check the register offsets against the HLS driver header
(`xfir_mmio_hw.h`) after synthesis.

In [28]:
# # Register offsets (from HLS synthesis report -- verify after build)
# CTRL_REG       = 0x00
# X_IN_OFFSET    = 0x18
# RETURN_OFFSET  = 0x10

# def float_to_uint(f):
#     return struct.unpack('<I', struct.pack('<f', f))[0]

# def uint_to_float(u):
#     return struct.unpack('<f', struct.pack('<I', u & 0xFFFFFFFF))[0]

# def mmio_fir_one_sample(ip, x_val):
#     ip.write(X_IN_OFFSET, float_to_uint(x_val))
#     ip.write(CTRL_REG, 0x01)                    # ap_start
#     while (ip.read(CTRL_REG) & 0x02) == 0:      # wait ap_done
#         pass
#     return uint_to_float(ip.read(RETURN_OFFSET))

In [ ]:
# y_mmio = np.zeros(N_SAMPLES, dtype=np.float32)

# t0 = time.perf_counter()y_mmio = np.zeros(N_SAMPLES, dtype=np.float32)

# t0 = time.perf_counter()
# for i in range(N_SAMPLES):
#     y_mmio[i] = mmio_fir_one_sample(fir_mmio_ip, float(x[i]))
# t_mmio = time.perf_counter() - t0

# print(f"MMIO: {N_SAMPLES} samples in {t_mmio*1e3:.2f} ms "
#       f"({t_mmio/N_SAMPLES*1e6:.1f} us/sample)")
# for i in range(N_SAMPLES):
#     y_mmio[i] = mmio_fir_one_sample(fir_mmio_ip, float(x[i]))
# t_mmio = time.perf_counter() - t0

# print(f"MMIO: {N_SAMPLES} samples in {t_mmio*1e3:.2f} ms "
#       f"({t_mmio/N_SAMPLES*1e6:.1f} us/sample)")

## 4. DMA Streaming FIR

Data path: PS $\rightarrow$ DMA TX $\rightarrow$ AXI-Stream $\rightarrow$ FIR $\rightarrow$ AXI-Stream $\rightarrow$ DMA RX $\rightarrow$ PS

The CPU only sets up the transfer and waits -- the FPGA processes all samples autonomously.

In [40]:
in_buf  = allocate(shape=(N_SAMPLES,), dtype=np.int32)
out_buf = allocate(shape=(N,),         dtype=np.int32)

np.copyto(in_buf, full_input_fixed)
out_buf[:] = 0

matmul_stream.write(0x10, 0x01)     # N = 1 input vector
matmul_stream.write(0x00, 0x01)  # ap_start

t0 = time.perf_counter()
dma.sendchannel.transfer(in_buf)
dma.recvchannel.transfer(out_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()
t_dma = time.perf_counter() - t0

y_dma = from_fixed(np.array(out_buf))
print(f"DMA done in {t_dma*1e3:.2f} ms")

DMA done in 4.58 ms


In [41]:
y_expected = output_vector.reshape(N,)
print(full_input)
for i in range(N):
    print(f"y[{i}] = {y_dma[i]:.6f}  expected = {y_expected[i]:.6f}")

err = np.linalg.norm(y_dma - y_expected)
print(f"\nL2 error: {err:.6e}")
print("PASS" if err < 1e-4 else "FAIL")

[-8.65208879e-02  3.30221169e-02 -2.53139645e-01  1.36383161e-01
 -1.07412279e-01  1.68809909e-02 -6.89118654e-02  3.34445871e-02
 -4.87929992e-02 -1.50510967e-01 -1.36936575e-01  1.51975602e-01
 -4.20161225e-02  8.79609957e-02  5.78107946e-02  2.31964886e-02
  1.61429912e-01 -1.87510684e-01  4.12395447e-02  2.60805469e-02
 -5.74059458e-03 -1.66989267e-02  1.23172425e-01 -5.70799634e-02
 -3.18753645e-02 -1.56941935e-01  1.40219787e-02  3.92278880e-02
 -4.90910374e-02  6.06634691e-02 -8.16670246e-03  4.20396999e-02
 -5.81795946e-02  1.60494328e-01 -1.24244981e-01  1.33536324e-01
  1.29681110e-01  1.73121858e-02  1.20346472e-01  6.82864431e-03
  8.29368606e-02 -8.12309086e-02 -2.01881211e-02  5.76537661e-02
 -1.89764738e-01 -8.74924287e-03  6.68828711e-02  2.29133666e-01
 -1.79447889e-01 -1.81790739e-01 -4.78202775e-02 -5.30312732e-02
  1.42374858e-01  2.97476146e-02 -9.88338292e-02  8.48591048e-03
 -2.09957920e-02 -1.63231462e-01 -1.71888068e-01 -6.72187936e-03
  3.27329412e-02 -1.25656

## 5. AXI Timer (cycle-accurate HW timing)

The AXI Timer counts PL clock cycles. This measures only the
HW processing time (including DMA transfer), without Python overhead.

In [ ]:
TCSR0, TLR0, TCR0 = 0x00, 0x04, 0x08
FCLK_MHZ = 100.0

def timer_start(tmr):
    tmr.write(TLR0, 0)
    tmr.write(TCSR0, 0x020)   # load
    tmr.write(TCSR0, 0x080)   # enable, count up

def timer_stop(tmr):
    cycles = tmr.read(TCR0)
    tmr.write(TCSR0, 0x000)
    return cycles
in_buf.freebuffer()
out_buf.freebuffer()

In [ ]:
# Re-run DMA test with HW timer
np.copyto(in_buf, x)
out_buf[:] = 0

matmul_stream.write(0x10, 0x01)
matmul_stream.write(0x00, 0x01)

timer_start(hw_timer)
dma.sendchannel.transfer(in_buf)
dma.recvchannel.transfer(out_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()
cycles = timer_stop(hw_timer)

print(f"HW timer: {cycles} cycles = {cycles/FCLK_MHZ:.1f} us @ {FCLK_MHZ:.0f} MHz")

## 6. Comparison

In [ ]:
print(f"{'Method':<12} {'Time (ms)':>10} {'Speedup':>10}")
print("-" * 34)
print(f"{'MMIO':<12} {t_mmio*1e3:>10.2f} {'1.0x':>10}")
print(f"{'DMA':<12} {t_dma*1e3:>10.2f} {t_mmio/t_dma:>9.1f}x")
print(f"{'HW cycles':<12} {cycles/FCLK_MHZ/1e3:>10.3f} {'(PL only)':>10}")
print()

max_diff = np.max(np.abs(y_dma - y_mmio))
print(f"Max |DMA - MMIO| = {max_diff:.2e}")
assert max_diff < 1e-4, "Output mismatch!"

## 7. Plot Results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(t * 1e3, x, label="Input (50+300 Hz)")
axes[0].set_ylabel("Amplitude")
axes[0].set_title("FIR Filter Input")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t * 1e3, y_dma, label="DMA output")
axes[1].plot(t * 1e3, y_mmio, '--', label="MMIO output", alpha=0.7)
axes[1].set_xlabel("Time (ms)")
axes[1].set_ylabel("Amplitude")
axes[1].set_title("FIR Filter Output (300 Hz suppressed)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Cleanup

In [ ]:
in_buf.freebuffer()
out_buf.freebuffer()